# Formula 1 - Modelling - Random Forest

### Import Relevant Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import requests
import bs4
from bs4 import BeautifulSoup
import time
from tqdm import tqdm
import math

In [ ]:
from collections import defaultdict
import regex as re
import datetime

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression, Perceptron, Ridge, Lasso, ElasticNet, RidgeCV, LassoCV, ElasticNetCV
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, BaggingRegressor
from sklearn import svm
from sklearn import metrics
from sklearn.metrics import r2_score

import xgboost
from xgboost import XGBRegressor

import joblib

# Load The Data

In [ ]:
driver_standings = pd.read_csv('./CSV/driver_standings.csv').drop(['Unnamed: 0'], axis=1)

In [ ]:
constructor_standings = pd.read_csv('./CSV/constructor_standings.csv').drop(['Unnamed: 0'], axis=1)

In [ ]:
main_df = pd.read_csv('./CSV/main_df.csv').drop(['Unnamed: 0'], axis=1)

In [ ]:
main_df.head()

In [ ]:
# age variable committed to csv as datetime type, loaded as a string. Reformat to remove ' days' from instances.

def day_split(age):
    return int(age.split(' ')[0])

main_df.ageDuringRace = main_df.ageDuringRace.apply(day_split)

In [ ]:
main_df.dtypes

In [ ]:
for col in main_df.columns:
    print('>>'+col)
    print(main_df[col].unique())

In [ ]:
race_dps = []

for years in np.array(main_df.season.unique()):
    race_dps.extend([(years, race_name) if race_name != '70th-anniversary' 
                     else (race_name.split('-')[0], race_name.split('-')[1], years)
                     for race_name in 
                     list(main_df[main_df.season == years]['race_name'].unique())])
    race_dps.extend([(years, 'usa') for race_name in
                 list(main_df[main_df.season == years]['race_name'].unique()) if race_name=='us'])
    race_dps.extend([(years, 'mexico') for race_name in
                 list(main_df[main_df.season == years]['race_name'].unique()) if race_name=='mexican'])

# Modelling

### Train-Test Split on pre-2020 to predict 2020 onwards

In [ ]:
# # Load a saved model

# pipe = joblib.load('ignore/models/f1model.pkl')
# model = joblib.load('ignore/models/f1model_XGB.pkl')
# model = joblib.load('ignore/models/f1model_RFR.pkl')

In [ ]:
main = main_df.copy()

train = main[main.season<2020]
test = main[main.season==2020]

r_train = train.pop('finish_position')
y_train = train.pop('filled_splits')
p_train = train.pop('points')
X_train = train
r_test = test.pop('finish_position')
y_test = test.pop('filled_splits')
X_test = test
p_test = test.pop('points')

In [ ]:
# Natural Language Processing
cvec_weather = CountVectorizer(stop_words='english', binary=True)#strip_words

# Dummification
ohe_direction = OneHotEncoder(handle_unknown='ignore')
ohe_country = OneHotEncoder(handle_unknown='ignore')
ohe_locality = OneHotEncoder(handle_unknown='ignore')
ohe_type = OneHotEncoder(handle_unknown='ignore')
ohe_season = OneHotEncoder(handle_unknown='ignore')
ohe_round = OneHotEncoder(handle_unknown='ignore')
ohe_qual_pos = OneHotEncoder(handle_unknown='ignore')
ohe_grid_pos = OneHotEncoder(handle_unknown='ignore')
ohe_racename = OneHotEncoder(handle_unknown='ignore')

# Standardisation
scaler_q_mean = StandardScaler(with_mean=False)
scaler_q_best = StandardScaler(with_mean=False)
scaler_q_worst = StandardScaler(with_mean=False)
scaler_length = StandardScaler(with_mean=False)
scaler_age = StandardScaler(with_mean=False)

In [ ]:
col_trans = ColumnTransformer([
                               ('weather', cvec_weather, 'weather'),
                               ('direction', ohe_direction, ['direction']),
                               ('country', ohe_country, ['country']),
                               ('locality', ohe_locality, ['locality']),
                               ('type', ohe_type, ['type']),
                               ('season', ohe_season, ['season']),
                               ('round', ohe_round, ['round']),
                               ('qual_position', ohe_qual_pos, ['qual_position']),
                               ('grid', ohe_grid_pos, ['grid']),
                               ('race_name', ohe_racename, ['race_name']),
                               ('q_mean', scaler_q_mean, ['q_mean']),
                               ('q_best', scaler_q_best, ['q_best']),
                               ('q_worst', scaler_q_worst, ['q_worst']),
                               ('length', scaler_length, ['length']),
                               ('ageDuringRace', scaler_age, ['ageDuringRace'])
                              ], 
                             )

In [ ]:
# weat = main_df.weather
# c = CountVectorizer(stop_words='english', binary=True).fit(weat)
# pd.DataFrame(c.transform(weat).toarray(), columns=c.vocabulary_)

In [ ]:
# a selection of some of the different models run to be called on for testing.

lin = LinearRegression(fit_intercept=False)
# pol = PolynomialFeatures(degree=3)
dtr = DecisionTreeRegressor()
# knr = KNeighborsRegressor()
# rfr = RandomForestRegressor(n_estimators=10000, n_jobs=-2, verbose=1, criterion='mae', ccp_alpha=0.1)
# rfr = RandomForestRegressor(n_estimators=1000, n_jobs=-2, verbose=1, criterion='mse', ccp_alpha=0.04)
mlp = MLPRegressor(hidden_layer_sizes=(15, 10, 6, 3, 1), solver='adam', activation='relu', alpha=1, max_iter=50000, random_state=1)
pcp = Perceptron(tol=10**(-6), penalty='l1', alpha=0.001, n_jobs=-2, shuffle=True)
# rcv = RidgeCV(alphas=[0.1, 0.01, 0.05, 0.001], fit_intercept=True, cv=4)
# rid = Ridge(alpha=0.1, solver='auto', max_iter=1000)
las = Lasso(alpha=3.2, max_iter=3000, tol=5220460544.0)
# lcv = LassoCV(alphas=[0.1, 0.01, 0.05, 0.001], fit_intercept=True, cv=4)
# eln = ElasticNet()
# ecv = ElasticNetCV(l1_ratio=0.001, n_alphas=150, n_jobs=-2, alphas=[0.1, 0.05, 0.01])
bgr = BaggingRegressor(base_estimator=lin, n_estimators=10000, n_jobs=-2, verbose=2)
gbr = GradientBoostingRegressor(n_estimators=10000, learning_rate=0.001, init=las, max_depth=30, verbose=1)


rfr = RandomForestRegressor(ccp_alpha=0.08,
                            criterion='mse',
                            max_depth=None,
                            max_leaf_nodes=None,
                            min_impurity_decrease=0.0,
                            min_samples_leaf=1,
                            min_samples_split=2,
                            min_weight_fraction_leaf=0.0,
                            n_estimators=1000,
                            verbose=1)

xgb = XGBRegressor(gamma=0.1,
                   learning_rate=0.2,
                   max_depth=6,
                   n_estimators=150,
                   reg_alpha=None,
                   reg_lambda=0.2,
                   subsample=1,
                   verbosity=1)

In [ ]:
# rfr.get_params()

In [ ]:
# params = {'ccp_alpha': [0.03, 0.032, 0.034, 0.036, 0.038, 0.04, 0.042, 0.044, 0.046, 0.048, 0.05],
#           'criterion': ['mse'],
#           'max_depth': [2, 4, None],
#           'max_features': ['auto'],
#           'max_leaf_nodes': [None, 2, 3],
#           'max_samples': [None],
#           'min_impurity_decrease': [0.0],
#           'min_impurity_split': [None],
#           'min_samples_leaf': [1, 2],
#           'min_samples_split': [2, 3],
#           'min_weight_fraction_leaf': [0.0],
#           'n_estimators': [1000],
#           'n_jobs': [-2],
#           'oob_score': [False],
#           'random_state': [0],
#           'verbose': [1],
#           'warm_start': [False]}

In [ ]:
# gs = GridSearchCV(rfr, params, verbose=1, n_jobs=-2, cv=5)

In [ ]:
# a selection of some of the different models fitted to pipelines for easy call to testing.

# pipe = make_pipeline(col_trans, lin)
# pipe = make_pipeline(col_trans, las)
# pipe = make_pipeline(col_trans, gbr)
# pipe = make_pipeline(col_trans, bgr)
# pipe = make_pipeline(col_trans, mlp)
# pipe = make_pipeline(col_trans, knr)

# pipe = make_pipeline(col_trans, rfr)
pipe = make_pipeline(col_trans, xgb)
# pipe = make_pipeline(col_trans, model)

In [ ]:
# pipe = make_pipeline(col_trans, gs)

All GridSearchCV & RandomizedSearchCV code has been warehoused in a separate file to increase presentability of this notebook. If you wish to see the code used, please don't hesitate to request the file.

In [ ]:
pipe.fit(X_train, y_train)

In [ ]:
print(pipe.score(X_train, y_train))
print(pipe.score(X_test, y_test))

In [ ]:
view_test = X_test.copy()
view_test['pred'] = pipe.predict(X_test)
view_test['true'] = y_test
view_test['true_finish_positions'] = r_test

test_indices = view_test.index

view_train = X_train.copy()
view_train['true'] = y_train
view_train['pred'] = pipe.predict(X_train)
view_train['true_finish_positions'] = r_train

train_indices = view_train.index

overall = pd.concat([view_test, view_train])
overall.sort_values(['season', 'round', 'pred'], inplace=True)




true_positions_test = []
indices_test = []
for race in race_dps:
    if race[1]=='anniversary':
        year = 2020
        circuit = '70th-anniversary'
    else:
        year = race[0]
        circuit = race[1]
    
    year_race = overall[(overall.season==year)&(overall.race_name==circuit)]

    def indexer(ind):
        return sorted(year_race.pred).index(ind)+1

    true_positions_test.extend(year_race.pred.apply(indexer))
    indices_test.extend(year_race.pred.apply(indexer).index)
positional_convert_test = pd.DataFrame(true_positions_test, index=indices_test).sort_index()
view_test['pred_positions'] = positional_convert_test




true_positions_train = []
indices_train = []
for race in race_dps:
    if race[1]=='anniversary':
        year = 2020
        circuit = '70th-anniversary'
    else:
        year = race[0]
        circuit = race[1]
    year_race = overall[(overall.season==year)&(overall.race_name==circuit)]

    def indexer(ind):
        return sorted(year_race.pred).index(ind)+1

    true_positions_train.extend(year_race.pred.apply(indexer))
    indices_train.extend(year_race.pred.apply(indexer).index)
positional_convert_train = pd.DataFrame(true_positions_train, index=indices_train).sort_index()
view_train['pred_positions'] = positional_convert_train




print(f'All Drivers Train Set R2 Score: {r2_score(view_train.true_finish_positions, view_train.pred_positions)}')
print(f'All Drivers Test Set R2 Score: {r2_score(view_test.true_finish_positions, view_test.pred_positions)}')




top_10_train = view_train[view_train.pred_positions<11]
print(f'Top 10 Finishers Test Set R2 Score: {r2_score(top_10_train.true_finish_positions, top_10_train.pred_positions)}')
top_10_test =  view_test[view_test.pred_positions<11]
print(f'Top 10 Finishers Test Set R2 Score: {r2_score(top_10_test.true_finish_positions, top_10_test.pred_positions)}')

In [ ]:
top_10_test['match'] = top_10_test['true_finish_positions']==top_10_test['pred_positions']
top_10_test['match_+/-1'] = (top_10_test['true_finish_positions']-top_10_test['pred_positions'])**2<=1
top_10_test['match_+/-2'] = np.sqrt((top_10_test['true_finish_positions']-top_10_test['pred_positions'])**2)<=2
top_10_test['match_+/-3'] = np.sqrt((top_10_test['true_finish_positions']-top_10_test['pred_positions'])**2)<=3

In [ ]:
top_10_test['match'].value_counts(normalize=True)

In [ ]:
top_10_test['match_+/-1'].value_counts(normalize=True)

In [ ]:
top_10_test['match_+/-2'].value_counts(normalize=True)

In [ ]:
top_10_test['match_+/-3'].value_counts(normalize=True)

In [ ]:
comparison_df = top_10_test.groupby('name')[['true_finish_positions', 'pred_positions']].agg(['sum','count'])

In [ ]:
comparison_df.head()

In [ ]:
comparison_df['true_average'] = comparison_df['true_finish_positions']['sum']/comparison_df['true_finish_positions']['count']
comparison_df['pred_average'] = comparison_df['pred_positions']['sum']/comparison_df['pred_positions']['count']

comparison_df.drop(['true_finish_positions', 'pred_positions'], axis=1).sort_values(['pred_average', 'true_average'])

In [ ]:
# custom function for groupby

In [ ]:
all_top_10 = pd.concat([top_10_train, top_10_test])
all_top_10 = all_top_10.sort_values(['season', 'round', 'pred_positions'])
all_top_10.reset_index(drop=True, inplace=True)
all_top_10 = all_top_10[['season', 'round', 'race_name', 'name', 'constructor', 'true_finish_positions', 'pred_positions']]
all_top_10.head(20)

### Further Exploration of the Results

In [ ]:
def points(pos):
    if pos==1: return 25
    if pos==2: return 18
    if pos==3: return 15
    if pos==4: return 12
    if pos==5: return 10
    if pos==6: return 8
    if pos==7: return 6
    if pos==8: return 4
    if pos==9: return 2
    if pos==10: return 1
    return 0

In [ ]:
def constructor(name):
    return main_df[main_df.name==name][constructor]

In [ ]:
view_test['points'] = view_test.pred_positions.apply(points)

In [ ]:
view_test.head()

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, plot_confusion_matrix

In [ ]:
print(confusion_matrix(view_test.true_finish_positions, view_test.pred_positions))

In [ ]:
np.diag(confusion_matrix(view_test.true_finish_positions, view_test.pred_positions)).sum()

What the figure above shows is the number of correctly predicted positions across the entire year.
40 positions equates to 2 of the 17 races correctly predicted (20 entrants per race for the 2020 season).

Alternatively, this can be summed up as a success race of 0.1176 or 11.76%.

In [ ]:
print(classification_report(view_test.true_finish_positions, view_test.pred_positions))

# Driver Standings

In [ ]:
driver_points = view_test[['name', 'points']]
predicted_standings_driver = driver_points.groupby('name').agg('sum').sort_values(by='points', ascending=False)
predicted_standings_driver.reset_index(inplace=True)
predicted_standings_driver.head()

In [ ]:
def driver_id(name):
    name = name.split()[1].strip(' ').lower()
    if name == 'fittipaldi':
        return 'pietro_fittipaldi'
    elif name == 'r?ikk?nen':
        return 'raikkonen'
    elif name == 'p?rez':
        return 'perez'
    elif name == 'h?lkenberg':
        return 'hulkenberg'
    elif name == 'magnussen':
        return 'kevin_magnussen'
    elif name == 'verstappen':
        return 'max_verstappen'
    else:
        return name

In [ ]:
predicted_standings_driver['driverId'] = predicted_standings_driver.name.apply(driver_id)
predicted_standings_driver.drop(['name'], axis=1, inplace=True)
predicted_standings_driver = predicted_standings_driver.reindex(columns=['driverId', 'points'])

In [ ]:
driver_comparison = driver_standings.drop(['position'], axis=1)
driver_comparison

In [ ]:
predicted_standings_driver

In [ ]:
driver_correlation_comparison = predicted_standings_driver.merge(driver_comparison, how='inner', on='driverId')
driver_correlation_comparison.columns=['driver', 'pred_points', 'true_points']
driver_correlation_comparison

In [ ]:
def pred_indexer(ind, df=driver_correlation_comparison):
    return sorted(df.pred_points, reverse=True).index(ind)+1
def true_indexer(ind, df=driver_correlation_comparison):
    return sorted(df.true_points, reverse=True).index(ind)+1

driver_correlation_comparison['pred_positions'] = driver_correlation_comparison.pred_points.apply(pred_indexer)
driver_correlation_comparison['true_positions'] = driver_correlation_comparison.true_points.apply(true_indexer)

In [ ]:
driver_correlation_comparison['Pos Error (Diff: Pred - True)'] = driver_correlation_comparison.pred_positions - driver_correlation_comparison.true_positions

In [ ]:
driver_correlation_comparison

In [ ]:
driver_correlation_comparison['Pos Error (Diff: Pred - True)'].sum()

In [ ]:
# true positive correct predictions

(driver_correlation_comparison.pred_positions == driver_correlation_comparison.true_positions).sum()

In [ ]:
# driver_correlation_comparison.to_markdown

In [ ]:
spearman_d = round(driver_correlation_comparison[['pred_points', 'true_points']].corr(method='spearman').iloc[0][1], 4)
pearson_d = round(driver_correlation_comparison[['pred_points', 'true_points']].corr(method='pearson').iloc[0][1], 4)
r2_d = round(r2_score(driver_correlation_comparison['true_points'], driver_correlation_comparison['pred_points']), 4)
mse_d = round(metrics.mean_squared_error(driver_correlation_comparison['true_points'], driver_correlation_comparison['pred_points']), 2)
rmse_d = round(np.sqrt(metrics.mean_squared_error(driver_correlation_comparison['true_points'], driver_correlation_comparison['pred_points'])), 3)

In [ ]:
print(f'Spearman Rank Correlation: {spearman_d}')
print(f'Pearson Correlation:       {pearson_d}')
print(f'R2 Score:                  {r2_d}')
print(f'Mean Squared Error:        {mse_d}')
print(f'Root Mean Squared Error:   {rmse_d}')

In [ ]:
# Spearman Rank Correlation: 0.9703
# Pearson Correlation:       0.9651
# R2 Score:                  0.925
# Mean Squared Error:        565.7
# Root Mean Squared Error:   23.784

In [ ]:
plt.subplots(figsize=(10, 10))
sns.regplot(data=driver_correlation_comparison, x='pred_points', y='true_points')
plt.plot([y_test.min(), 350], [y_test.min(), 350])
plt.title('Driver Standings Predicted vs. True')
plt.xlabel('Predicted Points')
plt.ylabel('True Points')

# Constructor Standings

In [ ]:
constructor_points = view_test[['constructor', 'points']]

In [ ]:
predicted_standings = constructor_points.groupby('constructor').agg('sum').sort_values(by='points', ascending=False)

In [ ]:
predicted_standings.reset_index(inplace=True)

In [ ]:
constructor_comparison = constructor_standings.drop(['position'], axis=1)

In [ ]:
correlation_comparison = predicted_standings.merge(constructor_comparison, how='inner', on='constructor')
correlation_comparison.columns=['constructor', 'pred_points', 'true_points']
correlation_comparison

In [ ]:
correlation_comparison['pred_positions'] = correlation_comparison.pred_points.apply(pred_indexer, df=correlation_comparison)
correlation_comparison['true_positions'] = correlation_comparison.true_points.apply(true_indexer, df=correlation_comparison)

In [ ]:
correlation_comparison

In [ ]:
# true positive correct predictions

(correlation_comparison.pred_positions == correlation_comparison.true_positions).sum()

In [ ]:
# correlation_comparison.to_markdown

In [ ]:
spearman = round(correlation_comparison[['pred_points', 'true_points']].corr(method='spearman').iloc[0][1], 4)
pearson = round(correlation_comparison[['pred_points', 'true_points']].corr(method='pearson').iloc[0][1], 4)
r2 = round(r2_score(correlation_comparison['true_points'], correlation_comparison['pred_points']), 4)
mse = metrics.mean_squared_error(correlation_comparison['true_points'], correlation_comparison['pred_points'])
rmse = round(np.sqrt(metrics.mean_squared_error(correlation_comparison['true_points'], correlation_comparison['pred_points'])), 3)

In [ ]:
print(f'Spearman Rank Correlation: {spearman}')
print(f'Pearson Correlation:       {pearson}')
print(f'R2 Score:                  {r2}')
print(f'Mean Squared Error:        {mse}')
print(f'Root Mean Squared Error:   {rmse}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
sns.regplot(data=correlation_comparison, x='pred_points', y='true_points')
x_label=('Predicted Points')
plt.plot([y_test.min(), 700], [y_test.min(), 700])
plt.title('Constructor Standings Predicted vs. True')
plt.xlabel('Predicted Points')
plt.ylabel('True Points')

In [ ]:
# # Save the model

# joblib.dump(pipe, 'ignore/models/f1model.pkl')